# NB_00_RP_43_SOURCE_EXTRACTION

This notebook converts the Becker engineering source into structured, reviewable RP_43 inputs and cumulative YAML specifications.

```text
Engineering source
        ↓
Manufacturing constraints
        ↓
Repeatable absorber process
        ↓
96-pixel detector module
        ↓
4,000-pixel instrument
        ↓
RP_43_A.yaml · RP_43_B.yaml · RP_43_C.yaml
```

This notebook is grounded in Dan Becker's presentation:

> *Achieving 1% Assay of Special Nuclear Materials in 2 Minutes with Microcalorimeter-Array Gamma-Ray Spectroscopy*  
> ARPA-E Fission Annual Meeting, October 1–2, 2025.

The notebook does not infer missing manufacturing details. Every extracted item remains traceable to a source page.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import json
import zipfile

try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pyyaml"],
        check=True,
    )
    import yaml

try:
    import pandas as pd
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pandas"],
        check=True,
    )
    import pandas as pd

NOTEBOOK_ID = "NB_00_RP_43_SOURCE_EXTRACTION"
NOTEBOOK_VERSION = "2.0.0"
REPOSITORY = "sensors-becker"

OUTPUT_DIRECTORY = Path("outputs/source_extraction/becker_2025_rp_43")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": REPOSITORY,
    "output_directory": str(OUTPUT_DIRECTORY),
}


## Source Identity

`source_file` records the reviewed PDF filename for provenance. This notebook does not open or parse the PDF during execution.


In [ ]:
SOURCE = {
    "source_id": "BECKER_2025_ARPA_E_MICROCALORIMETER_ASSAY",
    "title": (
        "Achieving 1% Assay of Special Nuclear Materials in 2 Minutes "
        "with Microcalorimeter-Array Gamma-Ray Spectroscopy"
    ),
    "author": "Dan Becker",
    "organization": "University of Colorado",
    "event": "ARPA-E Fission Annual Meeting",
    "date": "2025-10-01/2025-10-02",
    "source_file": "Daniel Becker (1).pdf",
    "page_count": 16,
    "engineering_object": "Microcalorimeter Array",
    "engineering_direction": (
        "Toward repeatable detector manufacturing and instrument-scale deployment."
    ),
}

SOURCE


## Extraction Schema

The RP_43 extraction categories are:

```text
manufacturing_constraint
manufacturing_refinement
assembly_requirement
module_target
instrument_target
deployment_requirement
```


In [ ]:
@dataclass(frozen=True)
class SourceExtraction:
    extraction_id: str
    category: str
    statement: str
    page: int
    metric: str | None = None
    value: float | int | str | None = None
    unit: str | None = None
    comparison_state: str | None = None
    note: str = ""

    def validate(self) -> None:
        allowed = {
            "manufacturing_constraint",
            "manufacturing_refinement",
            "assembly_requirement",
            "module_target",
            "instrument_target",
            "deployment_requirement",
        }
        if self.category not in allowed:
            raise ValueError(f"Unsupported category: {self.category}")
        if self.page < 1 or self.page > SOURCE["page_count"]:
            raise ValueError(f"Invalid source page: {self.page}")
        if not self.statement.strip():
            raise ValueError("statement is required")


## Source-Derived Extractions

The following entries are limited to manufacturing, assembly, scaling, and deployment statements supported by the presentation.


In [ ]:
EXTRACTIONS = [
    SourceExtraction(
        extraction_id="MC_001",
        category="manufacturing_constraint",
        statement="The pre-CURIE membrane architecture is finicky to assemble.",
        page=9,
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="MC_002",
        category="manufacturing_constraint",
        statement="The pre-CURIE membrane architecture is fragile.",
        page=9,
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="MC_003",
        category="manufacturing_constraint",
        statement="The project identifies an absorber manufacturing problem that must be solved.",
        page=15,
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="MR_001",
        category="manufacturing_refinement",
        statement="Use an all-silicon, membrane-free detector architecture.",
        page=10,
        comparison_state="refinement",
        note=(
            "Presented as forgiving to assemble, physically robust, and providing "
            "access to a wide range of thermal conductance G."
        ),
    ),
    SourceExtraction(
        extraction_id="MR_002",
        category="manufacturing_refinement",
        statement="Increase coupling to silicon.",
        page=14,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="AR_001",
        category="assembly_requirement",
        statement="Define requirements for product assembly.",
        page=15,
        comparison_state="requirement",
    ),
    SourceExtraction(
        extraction_id="MT_001",
        category="module_target",
        statement="Deploy a small 96-pixel module at the INL Analytical Laboratory.",
        page=15,
        metric="module_pixel_count",
        value=96,
        unit="pixels",
        comparison_state="near_term_target",
    ),
    SourceExtraction(
        extraction_id="IT_001",
        category="instrument_target",
        statement="Scale toward a 4,000-pixel instrument.",
        page=15,
        metric="instrument_pixel_count",
        value=4000,
        unit="pixels",
        comparison_state="long_term_target",
    ),
    SourceExtraction(
        extraction_id="IT_002",
        category="instrument_target",
        statement="Use the 4,000-pixel instrument to support measurements in 2 minutes.",
        page=15,
        metric="measurement_time",
        value=2,
        unit="minutes",
        comparison_state="long_term_target",
    ),
    SourceExtraction(
        extraction_id="DR_001",
        category="deployment_requirement",
        statement="Validate detector performance with relevant INL samples.",
        page=7,
        comparison_state="deployment",
    ),
    SourceExtraction(
        extraction_id="DR_002",
        category="deployment_requirement",
        statement="Build up microcalorimeter expertise for transition to market.",
        page=15,
        comparison_state="deployment",
    ),
]

for item in EXTRACTIONS:
    item.validate()

len(EXTRACTIONS)


## Review Extracted Engineering Content

Inspect this table before generating RP_43 YAML.


In [ ]:
extraction_table = pd.DataFrame(asdict(item) for item in EXTRACTIONS)
extraction_table[
    [
        "extraction_id",
        "category",
        "page",
        "metric",
        "value",
        "unit",
        "statement",
    ]
]


## Manufacturing-to-Deployment Sequence

```text
Pre-CURIE Architecture
        ↓
Repeatable Absorber Process
        ↓
96-Pixel Detector Module
        ↓
4,000-Pixel Instrument
```


In [ ]:
SEQUENCE_ROWS = [
    {
        "stage": "A",
        "input": "Absorber Manufacturing Problem",
        "output": "Repeatable Absorber Process",
        "support_a": "All-Silicon Architecture",
        "support_b": "Increase Si Coupling",
    },
    {
        "stage": "B",
        "input": "Repeatable Absorber Process",
        "output": "96-Pixel Detector Module",
        "support_a": "Assembly Requirements",
        "support_b": "INL Analytical Laboratory",
    },
    {
        "stage": "C",
        "input": "96-Pixel Detector Module",
        "output": "4,000-Pixel Instrument",
        "support_a": "INL Validation",
        "support_b": "1% Assay / 2 min",
    },
]

pd.DataFrame(SEQUENCE_ROWS)


## Candidate Source-Derived Reading Point

The visible dialogue carries source-specific manufacturing and deployment content. The repository grammar remains in metadata.


In [ ]:
READING_POINT_CANDIDATE = {
    "reading_point_id": "RP_43",
    "engineering_object": "Microcalorimeter Array",
    "source_id": SOURCE["source_id"],
    "dialogue": [
        {
            "order": "A",
            "concept": "Manufacturing Process",
            "title": "Detector Manufacturing: Microcalorimeters",
            "first_label": "Pre-CURIE Architecture",
            "second_label": "Repeatable Absorber Process",
            "supporting_context": [
                "All-Silicon Architecture",
                "Increase Si Coupling",
            ],
            "engineering_statement": (
                "Absorber manufacturing constraints direct a repeatable absorber process."
            ),
        },
        {
            "order": "B",
            "concept": "Detector Module",
            "title": "Detector Module: Microcalorimeters",
            "first_label": "Repeatable Absorber Process",
            "second_label": "96-Pixel Detector Module",
            "supporting_context": [
                "Assembly Requirements",
                "INL Analytical Laboratory",
            ],
            "engineering_statement": (
                "A repeatable absorber process supports detector-module assembly."
            ),
        },
        {
            "order": "C",
            "concept": "Instrument Scaling",
            "title": "Instrument Scaling: Microcalorimeters",
            "first_label": "96-Pixel Detector Module",
            "second_label": "4,000-Pixel Instrument",
            "supporting_context": [
                "INL Validation",
                "1% Assay / 2 min",
            ],
            "engineering_statement": (
                "Validated detector modules support scaling toward a 4,000-pixel instrument."
            ),
        },
    ],
}

READING_POINT_CANDIDATE


## Export Source Record and RP_43 Specifications

The notebook directly generates cumulative `RP_43_A.yaml`, `RP_43_B.yaml`, and `RP_43_C.yaml`.


In [ ]:
source_record = {
    "source": SOURCE,
    "extractions": [asdict(item) for item in EXTRACTIONS],
    "sequence": SEQUENCE_ROWS,
    "reading_point_candidate": READING_POINT_CANDIDATE,
}

json_path = OUTPUT_DIRECTORY / "becker_2025_rp_43_source_extraction.json"
yaml_path = OUTPUT_DIRECTORY / "becker_2025_rp_43_source_extraction.yaml"
review_path = OUTPUT_DIRECTORY / "becker_2025_rp_43_source_extraction.md"
candidate_path = OUTPUT_DIRECTORY / "RP_43_SOURCE_DERIVED.yaml"

json_path.write_text(
    json.dumps(source_record, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
yaml_path.write_text(
    yaml.safe_dump(source_record, sort_keys=False, allow_unicode=True, width=100),
    encoding="utf-8",
)
candidate_path.write_text(
    yaml.safe_dump(
        READING_POINT_CANDIDATE,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    ),
    encoding="utf-8",
)

review_lines = [
    "# RP_43 Source Extraction",
    "",
    f"**Source:** {SOURCE['title']}",
    "",
    "## Manufacturing-to-Deployment Sequence",
    "",
    "```text",
    "Pre-CURIE Architecture",
    "        ↓",
    "Repeatable Absorber Process",
    "        ↓",
    "96-Pixel Detector Module",
    "        ↓",
    "4,000-Pixel Instrument",
    "```",
    "",
    "## Source-Derived Extractions",
    "",
]
for item in EXTRACTIONS:
    review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(
    [
        "",
        "*Admissible generalizations trail leading specifications.*",
    ]
)

review_path.write_text("\n".join(review_lines) + "\n", encoding="utf-8")

REPOSITORY_GRAMMAR = [
    "Engineering Object specifies Engineering System.",
    "Engineering System produces Measured Engineering States.",
    "Measurement records Measured Engineering States.",
    "Measured Engineering States identify Engineering Constraints.",
    "Engineering Constraints direct Engineering Refinements.",
    "Engineering Refinements support Measured Engineering Improvement.",
    "Measured Engineering Improvement informs Leading Specifications.",
    "Leading Specifications direct Engineering Priorities.",
    "Engineering Priorities prepare Engineering Sessions.",
    "Engineering Sessions produce Engineering Records.",
    "Engineering Records support Engineering Reports.",
    "Engineering Reports support Repository Contributions.",
    "Repository Contributions support Repository Development.",
    "Repository Development supports Continued Specification.",
    "Continued Specification supports Engineering Object.",
]

STAGE_CONFIGURATION = {
    "A": {
        "notebook_id": "NB_43_A_ABSORBER_MANUFACTURING",
        "inherited_from": "NB_37_C_ENGINEERING_SESSIONS",
        "natural_foundation": "Detector Engineering Sessions",
        "engineering_objective": (
            "Specify a repeatable absorber process from source-derived manufacturing constraints."
        ),
        "forward_context": "Detector Module Assembly",
        "completion_status": "developing",
    },
    "B": {
        "notebook_id": "NB_43_B_DETECTOR_MODULE",
        "inherited_from": "NB_43_A_ABSORBER_MANUFACTURING",
        "natural_foundation": "Manufacturing Process",
        "engineering_objective": (
            "Support 96-pixel detector-module assembly from a repeatable absorber process."
        ),
        "forward_context": "Instrument Scaling",
        "completion_status": "developing",
    },
    "C": {
        "notebook_id": "NB_43_C_INSTRUMENT_SCALING",
        "inherited_from": "NB_43_B_DETECTOR_MODULE",
        "natural_foundation": "Detector Module",
        "engineering_objective": (
            "Support scaling from a validated 96-pixel module toward a 4,000-pixel instrument."
        ),
        "forward_context": "Instrument Deployment",
        "completion_status": "complete",
    },
}


def cumulative_rp_specification(stage: str) -> dict[str, Any]:
    stage_order = {"A": 1, "B": 2, "C": 3}
    if stage not in stage_order:
        raise ValueError(f"Unsupported RP_43 stage: {stage}")

    included_source = READING_POINT_CANDIDATE["dialogue"][: stage_order[stage]]
    included_dialogue = []

    for index, source_item in enumerate(included_source):
        dialogue_stage = source_item["order"]
        dialogue_status = (
            "candidate"
            if index == len(included_source) - 1
            else "admitted"
        )

        included_dialogue.append(
            {
                "order": dialogue_stage,
                "artifact_id": (
                    f"43_{dialogue_stage}_"
                    f"{source_item['concept'].lower().replace(' ', '_')}_trail"
                ),
                "concept": source_item["concept"],
                "title": source_item["title"],
                "first_label": source_item["first_label"],
                "second_label": source_item["second_label"],
                "supporting_context": source_item["supporting_context"],
                "engineering_statement": source_item["engineering_statement"],
                "status": dialogue_status,
            }
        )

    configuration = STAGE_CONFIGURATION[stage]

    return {
        "identity": {
            "notebook_id": configuration["notebook_id"],
            "reading_point": "RP_43",
            "stage": stage,
            "version": "1.0.0",
            "status": "candidate",
        },
        "reading_point": {
            "inherited_from": configuration["inherited_from"],
            "natural_foundation": configuration["natural_foundation"],
            "engineering_objective": configuration["engineering_objective"],
            "engineering_statements": [
                item["engineering_statement"]
                for item in included_dialogue
            ],
            "repository_grammar": REPOSITORY_GRAMMAR,
            "forward_context": configuration["forward_context"],
            "status": configuration["completion_status"],
        },
        "dialogue": included_dialogue,
        "engineering_object": SOURCE["engineering_object"],
        "engineering_direction": SOURCE["engineering_direction"],
        "source": SOURCE,
        "source_engineering_states": {
            category: [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == category
            ]
            for category in {
                "manufacturing_constraint",
                "manufacturing_refinement",
                "assembly_requirement",
                "module_target",
                "instrument_target",
                "deployment_requirement",
            }
        },
        "footer": "Admissible generalizations trail leading specifications.",
    }


rp_paths = {}
rp_specifications = {}

for stage in ("A", "B", "C"):
    specification = cumulative_rp_specification(stage)
    path = OUTPUT_DIRECTORY / f"RP_43_{stage}.yaml"

    path.write_text(
        yaml.safe_dump(
            specification,
            sort_keys=False,
            allow_unicode=True,
            width=100,
        ),
        encoding="utf-8",
    )

    rp_paths[stage] = path
    rp_specifications[stage] = specification

readme_path = OUTPUT_DIRECTORY / "RP_43_README.md"
readme_path.write_text(
    "# RP_43 — Manufacturing and Instrument Scaling\n\n"
    "A: pre-CURIE architecture → repeatable absorber process\n\n"
    "B: repeatable absorber process → 96-pixel detector module\n\n"
    "C: 96-pixel detector module → 4,000-pixel instrument\n\n"
    "Generated directly by NB_00_RP_43_SOURCE_EXTRACTION.\n",
    encoding="utf-8",
)

zip_path = OUTPUT_DIRECTORY / "NB_00_RP_43_SOURCE_EXTRACTION.zip"
bundle_paths = [
    json_path,
    yaml_path,
    review_path,
    candidate_path,
    rp_paths["A"],
    rp_paths["B"],
    rp_paths["C"],
    readme_path,
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in bundle_paths:
        archive.write(path, arcname=path.name)

generated = {
    "source_json": json_path,
    "source_yaml": yaml_path,
    "source_review": review_path,
    "reading_point_candidate": candidate_path,
    "rp_43_a": rp_paths["A"],
    "rp_43_b": rp_paths["B"],
    "rp_43_c": rp_paths["C"],
    "rp_43_readme": readme_path,
    "zip": zip_path,
}

generated


## Verification

This final cell verifies cumulative structure, exact titles, inheritance, source provenance, and statement agreement.


In [ ]:
expected_dialogue_counts = {"A": 1, "B": 2, "C": 3}

expected_titles = {
    "A": "Detector Manufacturing: Microcalorimeters",
    "B": "Detector Module: Microcalorimeters",
    "C": "Instrument Scaling: Microcalorimeters",
}

expected_inheritance = {
    "A": "NB_37_C_ENGINEERING_SESSIONS",
    "B": "NB_43_A_ABSORBER_MANUFACTURING",
    "C": "NB_43_B_DETECTOR_MODULE",
}

for label, path in generated.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    if path.stat().st_size <= 0:
        raise ValueError(f"Empty {label}: {path}")

for stage, expected_count in expected_dialogue_counts.items():
    loaded = yaml.safe_load(rp_paths[stage].read_text(encoding="utf-8"))

    identity = loaded["identity"]
    reading_point = loaded["reading_point"]
    dialogue = loaded["dialogue"]

    if identity["reading_point"] != "RP_43":
        raise ValueError(f"RP_43_{stage} has the wrong Reading Point identity")

    if identity["stage"] != stage:
        raise ValueError(f"RP_43_{stage} stage mismatch")

    if len(dialogue) != expected_count:
        raise ValueError(
            f"RP_43_{stage} contains {len(dialogue)} dialogues; "
            f"expected {expected_count}"
        )

    if dialogue[-1]["title"] != expected_titles[stage]:
        raise ValueError(f"RP_43_{stage} title mismatch")

    if reading_point["inherited_from"] != expected_inheritance[stage]:
        raise ValueError(f"RP_43_{stage} inheritance mismatch")

    if reading_point["engineering_statements"] != [
        item["engineering_statement"]
        for item in dialogue
    ]:
        raise ValueError(f"RP_43_{stage} statement mismatch")

    if loaded["source"]["source_id"] != SOURCE["source_id"]:
        raise ValueError(f"RP_43_{stage} source identity was not preserved")

print("RP_43 source extraction and YAML bundle: VERIFIED")
for label, path in generated.items():
    print(f"{label}: {path} ({path.stat().st_size} bytes)")

print()
print("Next:")
print("1. Extract RP_43_A.yaml, RP_43_B.yaml, and RP_43_C.yaml from the ZIP.")
print("2. Use each file as templates/RP_TEMPLATE.yaml.")
print("3. Run NB_TEMPLATE.ipynb for each cumulative stage.")

try:
    from google.colab import files
except ModuleNotFoundError:
    pass
else:
    files.download(str(zip_path))
